## **UNIQ Earth Sciences 2026**
### **Practical:** Climate Change and Coral Reefs

The purpose of this session is to explore a scientific problem, not to practice programming. These exercises do not require knowledge of programming (all of the code is already written, you may just need to modify some values) but if you need help, please ask! **To run a block of code, click on the code block and press the _play_ button at the top of the screen, or use the shortcut `ctrl+enter`.** You need to run code blocks sequentially for them to work properly.

<span style="color:Orangered ">**As you work through the practical, try and answer the core questions in orange**</span>. <span style="color:MediumPurple ">**Questions in purple are _extension_ questions: think about them if you have time left over**</span>. You do not need to hand in your answers and you will not be tested on them, but feel free to write down your answers if you want.

### **Overview**

In this practical session we will use a simple global climate _model_ to explore how the Earth's climate might change in the future, and how this might affect marine ecosystems. A climate model is a **mathematical representation of the climate system** which allows us to estimate how the state of the climate (e.g. temperature) responds to human actions (e.g. greenhouse gas emissions). Climate models are very important. Climate significantly impacts human livelihoods, and we need to know how our actions will affect the climate in the future. The Earth is far too large for us to easily perform experiments on the entire climate system (and even if it were possible, it wouldn't be ethical) so models are one of the only ways we can make these predictions about the future.

Today, we are going to use the `Finite-amplitude Impulse Response` model, or `FaIR` for short. Compared to 'state of the art' climate models, FaIR is actually still pretty simple, but it still allows us to make useful predictions for the future. FaIR allows the user to input emissions of greenhouse gases to estimate global mean temperature anomalies. For more information on FaIR, see the [documentation](https://docs.fairmodel.net/en/latest/).


***Acknowledgements:** Thanks to Peter Watson, Myles Allen and Helen Johnson for creating earlier versions of this practical, and to the `FaIR` development team for making the model available.*


### <span style="color:Orangered ">**Introductory question**</span>
<span style="color:Orangered ">**What is climate? Why is the climate currently changing?**</span>

## 1. How well can FaIR represent past climate change?

Before we can use a model to make predictions about the future, we need to test how well the model can do what it's supposed to do; in this case, predicting global temperature change given known emissions of greenhouse gasses. Here, **greenhouse gas emissions are our independent variable** and **global average temperature** is our depedent variable. We know how our greenhouse gas emissions have changed through time, and we have been measuring temperatures across the world for multiple centuries. **We can therefore test our model by giving it the known greenhouse gas emissions for the past few centuries, and seeing if it can correctly predict how the temperature has changed over the same timeframe.**

The code below sets up the model. You do not need to understand the code but, in short, we are giving the model the following information:
1. Which years we want to simulate (1850 - 2100)
2. How greenhouse gas emissions have changed historically, and different _scenarios_ for how they might change in the future depending on human actions.
3. A bunch of different model _configurations_ that we are going to test. The climate model takes many different parameters, which include things like how quickly heat gets transferred from the atmosphere to the surface ocean, and from the surface ocean to the deep ocean, and so on. **These parameters are all subject to uncertainty, and there is no single 'best' way to configure a climate model. Instead, we can build an _ensemble_ of configurations that represent our uncertainty in these parameters. We can propagate this uncertainty through the model by simulating climate using different configurations, and seeing how the predicted climate response varies across them**.

Click on the block of code below, and press the _play_ button or `ctrl+enter`. It will take a while to run. While it's running, `[*]` will appear to the top-left of the code block. Once the model set-up is complete, this will change to `[1]` (or possibly a different number if you are re-running it). This might take a short while to complete.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from fair import FAIR
from fair.io import read_properties
from fair.interface import fill, initialise
from fair.earth_params import seconds_per_year

# CREATE SIMULATION
f = FAIR() # Create a new model instance
f.define_time(1850, 2100, 1) # Run simulation from 1850 to 2100
seed = 123456 # Random seed

# Set up different SSP scenarios as SCENARIOS
f.define_scenarios(['ssp119', 'ssp126', 'ssp245', 'ssp370', 'ssp534-over', 'ssp585']) # Consider key CMIP6 SSPs

# Set up different GCM configurations as CONFIGS
df = pd.read_csv("../data/4xCO2_cummins_ebm3.csv")
# Set up exclusion for models with unrealistic TCR
exclude_model = ['CAS-ESM2', 'CNRM_CM6-1-HR', 'CNRM-ESM2-1', 'CIESM', 'CanESM5', 'E3SM-1', 'EC-Earth3', 'FIO-ESM-2-0', 'HadGEM3', 'INM-CM4-8', 'IPSL-CM6A-LR', 'NESM3', 'NorESM2-MM', 'TaiESM1', 'UKESM1-0-LL']
models = df['model'].unique()
models = [model for model in models if not np.any([excluded in model for excluded in exclude_model])]
configs = []
for imodel, model in enumerate(models):
    for run in df.loc[df['model']==model, 'run']:
        configs.append(f"{model}_{run}")
f.define_configs(configs)

# Set up different GHG species as SPECIES
species, properties = read_properties()
f.define_species(species, properties)

f.allocate() # Allocate input and output data

# CONFIGURE SIMULATION
# Import default GHG parameters into model
f.fill_species_configs()

# Import CMIP6 model-specific parameters
for config in configs:
    model, run = config.split('_')
    condition = (df['model']==model) & (df['run']==run)
    fill(f.climate_configs['ocean_heat_capacity'], df.loc[condition, 'C1':'C3'].values.squeeze(), config=config)
    fill(f.climate_configs['ocean_heat_transfer'], df.loc[condition, 'kappa1':'kappa3'].values.squeeze(), config=config)
    fill(f.climate_configs['deep_ocean_efficacy'], df.loc[condition, 'epsilon'].values[0], config=config)
    fill(f.climate_configs['gamma_autocorrelation'], df.loc[condition, 'gamma'].values[0], config=config)
    fill(f.climate_configs['sigma_eta'], df.loc[condition, 'sigma_eta'].values[0], config=config)
    fill(f.climate_configs['sigma_xi'], df.loc[condition, 'sigma_xi'].values[0], config=config)
    fill(f.climate_configs['stochastic_run'], True, config=config)
    fill(f.climate_configs['use_seed'], True, config=config)
    fill(f.climate_configs['seed'], seed, config=config)
    seed = seed + 399

# Set up SSP scenarios
f.fill_from_rcmip()
initialise(f.concentration, f.species_configs['baseline_concentration'])
initialise(f.forcing, 0)
initialise(f.temperature, 0)
initialise(f.cumulative_emissions, 0)
initialise(f.airborne_emissions, 0)

Once set-up is complete, run the code block below to _run_ the model.

In [ ]:
f.run()

As described above, we are going to test 44 different model _configurations_ to estimate the impacts of uncertainty in climate model parameters on the predicted warming, based on historical greenhouse gas emissions. Below, we will plot the model predictions for global temperature change from 1850 to 2020 for all model configurations. Each coloured line is a different configuration.

In [ ]:
# Plot warming under a specified climate scenario for all 66 model configurations
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.plot(f.timebounds, f.temperature.loc[dict(scenario='ssp245', layer=0)], label=f.configs)
ax.set_xlabel('Year')
ax.set_ylabel(r'Temperature anomaly vs pre-industrial ($^\circ$C)')
ax.set_xlim([1850, 2020])
ax.set_ylim([-0.5, 2.0])
ax.legend(frameon=False, ncol=2, fontsize=10, loc='lower center', bbox_to_anchor=(1.3, 0.07), title='Configuration')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

### <span style="color:Orangered ">Question 1.1</span>
<span style="color:Orangered ">**Although we are using a computer model rather than carrying out a 'real' experiment, this modelling experiment can be thought of as representing a hypothesis, namely that _the model reasonably represents the processes that translate greenhouse gas emissions into an effect on the Earth's climate_. In this experiment, what is/are the independent variable(s), and what is/are the dependent variable(s)?**</span>

### <span style="color:MediumPurple ">Question 1.2</span>
<span style="color:MediumPurple ">**These 44 model configurations differ by the parameters that were chosen that set the model behaviour. Two important parameters are the _heat capacities_ of the shallow and deep ocean. Will models with a higher heat capacity of the shallow ocean predict faster or slower warming? What about models with a higher heat capacity of the deep ocean? Which do you think is more important for warming over the past few decades?**</span>

As explained above, we don't know which configuration is 'correct', so the spread between models is a representation of uncertainty. **We call this model, or structural, uncertainty.** The above plot with 44 lines is a bit messy, so we could alternatively just plot 1 standard deviation (dark shaded) and 2 standard deviations (pale shaded) across the configurations, plus the mean across all configurations (bold line).

In [ ]:
# Calculate mean + stdev
mean = f.temperature.loc[dict(scenario='ssp245', layer=0)].mean(dim='config')
stdev = f.temperature.loc[dict(scenario='ssp245', layer=0)].std(dim='config')

# Plot warming under a specified climate scenario for all model configurations (multi-model mean + standard deviation[s])
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

ax.plot(f.timebounds, mean, c='k', linewidth=2)
ax.fill_between(f.timebounds, mean - 2*stdev, mean + 2*stdev, color='k', alpha=0.1, lw=0)
ax.fill_between(f.timebounds, mean - 1*stdev, mean + 1*stdev, color='k', alpha=0.2, lw=0)
ax.set_xlabel('Year')
ax.set_ylabel(r'Temperature anomaly vs pre-industrial ($^\circ$C)')
ax.set_xlim([1850, 2020])
ax.set_ylim([-0.5, 2.0])
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

### <span style="color:Orangered ">Question 1.3</span>
<span style="color:Orangered ">**How does the magnitude of the uncertainty in warming across the configurations compare to the mean simulated warming since 1850? In what decade did the magnitude of warming exceed inter-configuration uncertainty?**</span>

### <span style="color:MediumPurple ">Question 1.4</span>
<span style="color:MediumPurple ">**Apart from uncertainty in model parameters, what other sources of uncertainty might there be when simulating past climate using a model?**</span>

### <span style="color:MediumPurple ">Question 1.5</span>
<span style="color:MediumPurple ">**What do you think could be responsible for the short-lived cooling pulses in 1883, 1963, and 1991?**</span>

We will now assess how well this model (and its constituent configurations) reproduce _observed_ warming, based on the historical greenhouse gas emissions we have provided. We will plot the same figure again, but this time we will add observational data on top. The dataset we will use is HadCRUT5, which is a statistical estimate of global mean surface temperature anomalies since 1850, based purely on observations. Since statistical approaches are used to fill in gaps where observations are unavailable, the dataset also includes an estimate of uncertainty (1 standard deviation).

In [ ]:
# Import HadCRUT5 historical global mean surface temperature dataset
hadcrut5 = pd.read_csv("../data/gmt_HadCRUT5.csv")

# Replot the above figure, adding the observational data
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Model
ax.plot(f.timebounds, mean, c='k', linewidth=2, label='Model')
ax.fill_between(f.timebounds, mean - 2*stdev, mean + 2*stdev, color='k', alpha=0.1, lw=0)
ax.fill_between(f.timebounds, mean - 1*stdev, mean + 1*stdev, color='k', alpha=0.2, lw=0)

# Obs
ax.plot(hadcrut5.Year, hadcrut5['HadCRUT5 (degC)'], c='orangered', linewidth=2, linestyle='-', label='Observations')
ax.fill_between(hadcrut5.Year, hadcrut5['HadCRUT5 (degC)'] - hadcrut5['HadCRUT5 uncertainty'],
                hadcrut5['HadCRUT5 (degC)'] + hadcrut5['HadCRUT5 uncertainty'], color='orangered', alpha=0.2, lw=0)

ax.set_xlabel('Year')
ax.set_ylabel(r'Temperature anomaly vs pre-industrial ($^\circ$C)')
ax.legend(frameon=False)
ax.set_xlim([1850, 2020])
ax.set_ylim([-0.5, 2.0])
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

### <span style="color:Orangered ">Question 1.6</span>
<span style="color:Orangered ">**What is the point of comparing the model predictions to past (observed) climate?**</span>

### <span style="color:MediumPurple ">Question 1.7</span>
<span style="color:MediumPurple ">**Why is there observational uncertainty? How does it change with time, and why?**</span>

### <span style="color:Orangered ">Question 1.8</span>
<span style="color:Orangered ">**How successfully does the model reproduce observations? Does this model-observation comparison _support_ the hypothesis that _the model reasonably represents the processes that translate greenhouse gas emissions into an effect on the Earth's climate_?**</span>

### <span style="color:MediumPurple ">Question 1.9</span>
<span style="color:MediumPurple ">**Is the model-observation comparison sufficient to _confirm_ that the above hypothesis is true? What further tests might you want to perform?**</span>

### <span style="color:MediumPurple ">Question 1.10</span>
<span style="color:MediumPurple ">**Are you comfortable with using the model to predict future climate? What limitations might you want to keep in mind if we use the model for this purpose?**</span>

## 2. Predicting future climate change with FaIR

If the hypothesis described above (in Q1.9-10) is supported by evidence, we could use this model as a _predictive_ tool to explore how the Earth's climate might change in response to future (yet unrealised) greenhouse gas emissions. It's important to remember that, in contrast to Q1 where we were using the climate model as a hypothesis-testing tool, we are now using it in a very different context: to generate predictions that we cannot (immediately) test.

We don't know how greenhouse gas emissions will change in the future. In contrast to the physical climate system (which is, in principle, deterministic), future greenhouse gas emissions depend on future human behaviour, which in turn depends on complex social and economic processes. As a result, it is impossible for us to accurately predict how greenhouse gas emissions will change in the future (at least not as natural scientists). Instead, we can imagine a set of _hypothetical scenarios_ which describe how emissions _could_ change in the future depending on different socio-economic pathways the world might choose to go down.

These scenarios are shown in the image below. They are commonly called SSPs, or _shared socio-economic pathways_. We won't worry about the naming too much in this session*, but in short the number after the hyphen is related to the total amount of greenhouse gas emissions (e.g. SSP5-**8.5** represents more emissions than SSP1-**2.6**). 

**There are two important things to know about these SSP scenarios that are commonly misunderstood:**

1. **SSPs are not predictions, they are hypothetical scenarios** we use to investigate how the climate might change in the future _if_ we followed different greenhouse gas emission reduction strategies. For example, SSP2-4.5 is broadly in line with current policies for future emissions reductions, so might represent a 'middle of the road' future where we stick to existing policies but fail to make them more ambitious. SSP1-2.6 is an optimistic scenario that is broadly consistent with the goals of the Paris Climate Agreement to limit warming to 2 degrees C. This is useful because, even though we don't know what emmissions will look like in the future, by using these scenarios we can tell governments "if you keep emmissions at the current level, the climate will likely look like this" or "if you reduce emmissions by X%, the climate will instead look like that", and so on.

2. **SSP scenarios only describe human activities (greenhouse gas emmissions and land use), they do not describe the resulting climate change**. Climate change is predicted by models _using_ SSP scenarios (i.e. the SSP scenario is the input, and predicted climate change is the output). 

\* _If you want further information, you can read [this](https://www.carbonbrief.org/explainer-how-shared-socioeconomic-pathways-explore-future-climate-change/) explainer after the session (which was also included as optional UNIQ pre-reading)._

![SSPs overview](ssps.jpg)
_Figure 1:_ Overview of CMIP6 SSPs from [O'Neill et al. (2016)](https://gmd.copernicus.org/articles/9/3461/2016/).

### <span style="color:Orangered ">Question 2.1</span>
<span style="color:Orangered ">**Why aren't there any scenarios for the past?**</span>

### <span style="color:MediumPurple ">Question 2.2</span>
<span style="color:MediumPurple ">**Which scenario would result in the lowest _cumulative_ emmissions between now and 2100?**</span>

### <span style="color:Orangered ">Question 2.3</span>
<span style="color:Orangered ">**What do you think it means when a scenario goes below the x axis?**</span>

### <span style="color:Orangered ">Question 2.4</span>
<span style="color:Orangered ">**Which scenarios reach net zero by 2100?**</span>

You may have noticed that this uncertainty in future emissions represents another potential source of uncertainty for future climate predictions. We have already touched on _model uncertainty_ in Part 1 (due to uncertainty in model design). **This new source of uncertainty concerning future human behaviour is called _scenario uncertainty_.**

### <span style="color:Orangered ">Question 2.5</span>
<span style="color:Orangered ">**How does _scenario uncertainty_ change with time?**</span>

As explained about, SSPs describe _emissions_. To predict the climate change resulting from these SSP scenarios, we have to insert them into a climate model. The code below plots the climate change resulting from the 'middle of the road' SSP2-4.5 scenario. As in part 1, the historical observations are in red.

In [ ]:
# Choose scenario
scenario = 'ssp245' # Options: ssp119, ssp126, ssp245, ssp370, ssp534-over, ssp585

# Calculate mean + stdev
mean = f.temperature.loc[dict(scenario=scenario, layer=0)].mean(dim='config')
stdev = f.temperature.loc[dict(scenario=scenario, layer=0)].std(dim='config')

# Plot warming under a specified climate scenario for all model configurations (multi-model mean + standard deviation[s])
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.plot(f.timebounds, mean, c='k', linewidth=2, label='Model')
ax.plot(hadcrut5.Year, hadcrut5['HadCRUT5 (degC)'], c='orangered', linewidth=2, linestyle='-', label='Observations')
ax.fill_between(f.timebounds, mean - 2*stdev, mean + 2*stdev, color='k', alpha=0.1, lw=0)
ax.fill_between(f.timebounds, mean - 1*stdev, mean + 1*stdev, color='k', alpha=0.2, lw=0)
ax.legend(frameon=False)
ax.set_xlabel('Year')
ax.set_ylabel(r'Temperature anomaly vs pre-industrial ($^\circ$C)')
ax.set_xlim([1850, 2100])
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

Recall from section 1 that _model uncertainty_ refers to the spread between the 44 different model configurations we're testing, due to uncertainty in how to represent the climate system in a model.

### <span style="color:Orangered ">Question 2.6</span>
<span style="color:Orangered ">**How does _model uncertainty_ change with time?**</span>

### <span style="color:Orangered ">Question 2.7</span>
<span style="color:Orangered ">**By modifying line 2 in the code above, plot the predicted climate change arising from the following SSP scenarios: SSP1-1.9, SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-3.4 overshoot, and SSP5-8.5 By referring to figure 1 in section 1, what is the relationship between cumulate CO2 emissions and 21st century warming?**</span>

### <span style="color:Orangered ">Question 2.8</span>
<span style="color:Orangered ">**The Paris Climate Agreement represented a legally binding commitment by countries to limit warming to 2$^\circ$C. Using this climate model, which emmissions scenarios are compatible with the goals of the Paris Climate Agreement, and how confident can we be?**</span>

### <span style="color:MediumPurple ">Question 2.9</span>
<span style="color:MediumPurple ">**Assuming 21st century emmissions follow SSP2-4.5 (broadly following current policies), how likely is it that warming exceeds 3$^\circ$C by 2100?**</span>

### <span style="color:MediumPurple ">Question 2.10</span>
<span style="color:MediumPurple ">**The SSP scenarios are now more than a decade old. In 2025, annual CO2 emissions were 38.2 GtCO2. Considering figure 1 and the simulations above, is it still possible to limit warming to 2$^\circ$C by 2100?**</span>

![SSPs overview](uncertainty.png)

_Figure 2:_ Sources of uncertainty in 21st century climate prediction. <span style="color:Orangered ">Orange: internal climate variability (don't worry about this).</span> <span style="color:Blue ">Blue: Model uncertainty (see Q1.2).</span> <span style="color:Green ">Green: Scenario uncertainty (see Q2.4).</span> Figure from IPCC AR5.

### <span style="color:MediumPurple ">Question 2.11</span>
<span style="color:MediumPurple ">**We have so far discussed two major sources of uncertainty in future climate prediction: model uncertainty (due to uncertainty in the equations that describe the Earth's climate) and scenario uncertainty (due to uncertainty in how greenhouse gas emissions will change in the future). From your simulations above, you can see how the size of these uncertainties change with time (e.g. Q2.5 and 2.6). The relative magnitude of these uncertainties (plus another source called internal variability, which we will ignore in this question) are compared in figure 2, above. Considering figure 2, to what extent can further improvements in climate science increase confidence in warming predictions for 2050? What about 2100?**</span>

### <span style="color:MediumPurple ">Question 2.12</span>
<span style="color:MediumPurple ">**The 'internal variability' source of uncertainty in Figure 2 refers to stochastic variations in the Earth's climate. For example, you might have heard of El Nino, which involves a characteristic pattern of temperature, rainfall, and wind anomalies occuring every 2-7 years. This is an example of internal variability. Why is this source of uncertainty enormous for near-term (<10 years) climate prediction, but negligible for climate prediction in the second half of the century?**</span>

## 3. Predicting coral reef futures

Corals are marine organisms that are very sensitive to the ocean temperature. To predict how corals might fare under climate change, we can try modelling their population dynamics. In the absence of environmental stress, a common model for simulating population size is called _logistic growth_:

$$\frac{dN}{dt}=rN(1-N)$$

where $N$ is the fractional population size (between 0 and 1), and $r$ is the population growth rate scaling parameter. 

### <span style="color:Orangered ">Question 3.1</span>
<span style="color:Orangered ">**Under the logistic growth model, when is the growth rate $dN/dt$ minimum and maximum? What is the ecological interpretation of this?**</span>

This model is not yet a complete description of coral population dynamics because it doesn't incorporate mortality arising from environmental stress. To account for this, we can add an additional term to the equation above:

$$\frac{dN}{dt}=rN(1-N)-mN(T(t)-T_m)\cdot\delta(T(t)>T_m)$$

where $T(t)$ is the temperature (itself a function of time), $m$ is the mortality rate scaling parameter, $T_m$ is a temperature above which temperature-induced mortality begins, and $\delta(x)=1$ if $x$ is True or otherwise $\delta(x)=0$ if $x$ is False.

### <span style="color:Orangered ">Question 3.2</span>
<span style="color:Orangered ">**What is the relationship between the mortality rate and temperature in this model?**</span>

While you could try using your A Level Maths to solve the above equation, you will quickly run into problems. Instead, we will use the computer to solve the equations for us, using the temperature predictions from our climate model simulations in sections 1-2.

In [ ]:
# MODEL PARAMETERS
r = 0.1 # Population growth rate scaling parameter (1/y)
m = 0.05 # Mortality rate scaling parameter (1/C*y)
Tm = 28 # Temperature-induced mortality thresholds (C)
scenario = 'ssp245' # SSP scenario. Options: ssp119, ssp126, ssp245, ssp370, ssp534-over, ssp585

# INITIAL + BOUNDARY CONDITIONS
base_tropical_sst = 27.5 # Pre-climate change tropical sea surface temperature
base_coral_cover = 1 # Assume coral population size was 100% before humans
sst = f.temperature.loc[dict(scenario=scenario, layer=0)] + base_tropical_sst
years = f.timebounds

# MODEL
# Initialise output array
N = np.nan*np.ones_like(sst) # Initialise output array
N[0, :] = base_coral_cover

# Define dNdt
def dNdt(_N, _T, r, m, Tm):
    # Evaluate dN/dt at N(t) subject to T(t) and r, m, Tm
    delta = np.where(_T > Tm, 1.0, 0.0)
    return r*_N*(1 - _N) - m*_N*(_T - Tm)*delta
    
# Naive RK1 implementation to solve the simple coral population model.
dt = 1.0
for i, time in enumerate(years[1:]):
    _N_next = N[i, :] + dNdt(N[i, :], sst[i, :], r, m, Tm)*dt
    _N_next = np.where(_N_next > 1.0, 1.0, _N_next)
    _N_next = np.where(_N_next < 0.0, 0.0, _N_next)
    N[i+1, :] = _N_next

# PLOT OUTPUT
mean_coral = N.mean(axis=1)*100
stdev_coral = N.std(axis=1)*100
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.fill_between(f.timebounds, mean_coral - 2*stdev_coral, mean_coral + 2*stdev_coral, color='k', alpha=0.1, lw=0)
ax.fill_between(f.timebounds, mean_coral - 1*stdev_coral, mean_coral + 1*stdev_coral, color='k', alpha=0.2, lw=0)
ax.plot(f.timebounds, mean_coral, c='k', linewidth=2)
ax.set_xlabel('Year')
ax.set_ylabel('Coral cover (%)')
ax.set_xlim([1950, 2100])
ax.set_ylim([0.0, 100.0])
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

### <span style="color:Orangered ">Question 3.3</span>
<span style="color:Orangered ">**Under this default configuration, which SSP scenarios are consistent with coral survival to the end of the century? For the purpose of this question, we will assume that a population has 'collapsed' when $N<0.05$.**</span>

### <span style="color:MediumPurple ">Question 3.4</span>
<span style="color:MediumPurple ">**There are three key parameters in this ecological model: $r$, $m$, and $T_m$. How might we decide which values are appropriate for these parameters?**</span>

### <span style="color:MediumPurple ">Question 3.5</span>
<span style="color:MediumPurple ">**Try modifying these parameters in the code above (lines 4-6). Which parameter(s) are most important in determining whether corals survive to the end of the century in this model?**</span>

Let's remind ourselves of the equation that defines this model:

$$\frac{dN}{dt}=rN(1-N)-mN(T(t)-T_m)\cdot\delta(T(t)>T_m)$$

There model assumes that the temperature-induced mortality threshold, $T_m$ is a constant. In fact, this threshold may evolve through time due to adaptation and evolution. We could introduce this adaptive process through a new equation describing how $T_m$ changes through time:

$$\frac{dT_m}{dt} = a\left(T(t) - T_m(t)\right)$$

where $a$ is the adaptation rate scaling parameter.

### <span style="color:Orangered ">Question 3.6</span>
<span style="color:Orangered ">**Describe and explain this new equation and the assumption(s) it represents.**</span>

### <span style="color:Orangered ">Question 3.7</span>
<span style="color:Orangered ">**What do you predict will happen to modelled coral populations when $a$ is increased?**</span>

The code below includes this new adaptation mechanic. To begin with, $a=0.0$ y$^{-1}$ so the model behaves the same as before.

In [ ]:
# MODEL PARAMETERS
r = 0.1 # Population growth rate scaling parameter (1/y)
m = 0.05 # Mortality rate scaling parameter (1/C*y)
a = 0.00 # Adaptation rate scaling parameter (1/y)
adaptation_offset = 0.05 # Difference between Tm and local temperature at equilibrium (C)
scenario = 'ssp245' # SSP scenario. Options: ssp119, ssp126, ssp245, ssp370, ssp534-over, ssp585

# INITIAL + BOUNDARY CONDITIONS
base_tropical_sst = 27.5 # Pre-climate change tropical sea surface temperature
base_Tm = 28
base_coral_cover = 1 # Assume coral population size was 100% before humans
sst = f.temperature.loc[dict(scenario=scenario, layer=0)] + base_tropical_sst
years = f.timebounds

# MODEL
N = np.nan*np.ones_like(sst) # Initialise output array
Tm = np.nan*np.ones_like(sst) # Initialise Tm array
N[0, :] = base_coral_cover
Tm[0, :] = base_Tm

# Define dNdt
def dNdt(_N, _T, r, m, _Tm):
    # Evaluate dN/dt at N(t) subject to T(t) and r, m, Tm
    delta = np.where(_T > _Tm, 1.0, 0.0)
    return r*_N*(1 - _N) - m*_N*(_T - _Tm)*delta

def dTmdt(_Tm, _T, a):
    # Evaluate dTm/dt at Tm(t) subject to T(t)
    return a*(_T - _Tm + adaptation_offset) # This comment has intentionally been made useless for Q3.11!

# Naive RK1 implementation to solve the simple coral population model.
dt = 1.0
for i, time in enumerate(years[1:]):
    _Tm_next = Tm[i, :] + dTmdt(Tm[i, :], sst[i, :], a) 
    
    _N_next = N[i, :] + dNdt(N[i, :], sst[i, :], r, m, _Tm_next)*dt
    _N_next = np.where(_N_next > 1.0, 1.0, _N_next)
    _N_next = np.where(_N_next < 0.0, 0.0, _N_next)
    
    N[i+1, :] = _N_next
    Tm[i+1, :] = _Tm_next

# PLOT OUTPUT
mean_coral = N.mean(axis=1)*100
stdev_coral = N.std(axis=1)*100
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.fill_between(f.timebounds, mean_coral - 2*stdev_coral, mean_coral + 2*stdev_coral, color='k', alpha=0.1, lw=0)
ax.fill_between(f.timebounds, mean_coral - 1*stdev_coral, mean_coral + 1*stdev_coral, color='k', alpha=0.2, lw=0)
ax.plot(f.timebounds, mean_coral, c='k', linewidth=2)
ax.set_xlabel('Year')
ax.set_ylabel('Coral cover (%)')
ax.set_xlim([1950, 2100])
ax.set_ylim([0.0, 100.0])
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

### <span style="color:Orangered ">Question 3.8</span>
<span style="color:Orangered ">**With $r$ and $m$ at their default values ($r=0.1$ y$^{-1}$ and $m=0.05$ $^\circ$ C $^{-1}$ y$^{-1}$), increase $a$ to $0.01$ y$^{-1}$. Was your answer to Q3.7 correct?**</span>

### <span style="color:Orangered ">Question 3.9</span>
<span style="color:Orangered ">**How big does $a$ have to be to maintain coral cover above 25% under SSP2-4.5? What about SSP1-2.6?**</span>

### <span style="color:MediumPurple ">Question 3.10</span>
<span style="color:MediumPurple ">**How might we identify what a realistic value of $a$ is?**</span>

### <span style="color:MediumPurple ">Question 3.11</span>
<span style="color:MediumPurple ">**What is the ecological interpretation of the `adaptation_offset` variable in the above code?**</span>

### <span style="color:MediumPurple ">Question 3.12</span>
<span style="color:MediumPurple ">**To what extent do you think this is a useful model for predicting the response of corals to climate change? How might you test this?**</span>